# HQC-PINN: Hybrid Quantum-Classical Physics-Informed Neural Network

**Demonstration Notebook** for the paper:
> *Variational Quantum Physics-Informed Neural Networks for Hydrological PDE-Constrained Learning with Inherent Uncertainty Quantification*

This notebook demonstrates:
1. Building the HQC-PINN architecture (PennyLane + PyTorch)
2. Quantum circuit visualization and gate counting
3. Training with physics-informed loss
4. Uncertainty quantification via quantum measurement sampling
5. Comparison with classical PINN baseline

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import pennylane as qml
from torch.utils.data import DataLoader, random_split

from src.models import HQCPINN, ClassicalPINN, VariationalQuantumCircuit
from src.data.dataset import FloodDataset, FLOOD_CLASSES, TOTAL_FEATURES
from src.training import PhysicsInformedLoss, HQCPINNTrainer
from src.evaluation.metrics import compute_classification_metrics

torch.manual_seed(42)
np.random.seed(42)

print(f"PennyLane version: {qml.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Feature dimension: {TOTAL_FEATURES}")
print(f"Classes: {FLOOD_CLASSES}")

## 1. Quantum Circuit Architecture

The VQC uses angle encoding (R_Y gates) followed by a hardware-efficient ansatz with alternating R_Y/R_Z rotations and CNOT entanglement.

In [ ]:
# Create and inspect the VQC
vqc = VariationalQuantumCircuit(n_qubits=8, n_layers=3)

print("=== Quantum Circuit Resource Estimates ===")
gate_info = vqc.gate_count()
for key, value in gate_info.items():
    print(f"  {key:25s}: {value}")

print(f"\nHilbert space dimension: 2^8 = {2**8}")
print(f"Trainable parameters: {vqc.n_params}")
print(f"Parameter formula: 2 × n_qubits × n_layers = 2 × 8 × 3 = {2*8*3}")

In [ ]:
# Visualize the quantum circuit
dev = qml.device('default.qubit', wires=8)

@qml.qnode(dev)
def demo_circuit(inputs, params):
    from src.models.quantum_circuit import angle_encoding, hardware_efficient_ansatz
    wires = list(range(8))
    angle_encoding(inputs, wires)
    reshaped = params.reshape(3, 8, 2)
    hardware_efficient_ansatz(reshaped, wires, 3)
    return [qml.expval(qml.PauliZ(w)) for w in wires]

dummy_input = torch.randn(8)
dummy_params = torch.randn(48)

fig, ax = qml.draw_mpl(demo_circuit)(dummy_input, dummy_params)
fig.set_size_inches(16, 6)
fig.suptitle('HQC-PINN Variational Quantum Circuit (8 qubits, 3 layers)', fontsize=14)

## 2. HQC-PINN Model Architecture

In [ ]:
# Build the full HQC-PINN
hqc_model = HQCPINN(
    input_dim=TOTAL_FEATURES,  # 25 multi-modal features
    n_qubits=8,
    n_layers=3,
    n_classes=4,               # 4 flood severity levels
    hidden_pre=64,
    hidden_post=32,
)

# Build classical baseline
cpinn_model = ClassicalPINN(
    input_dim=TOTAL_FEATURES,
    hidden_dims=[256, 128, 64],
    n_classes=4,
)

print("=== Parameter Comparison (Table III) ===")
hqc_params = hqc_model.count_parameters()
cpinn_params = cpinn_model.count_parameters()

print(f"\nHQC-PINN:")
print(f"  Classical pre-net:  {hqc_params['classical_pre']:,}")
print(f"  Quantum VQC:        {hqc_params['quantum_vqc']:,}")
print(f"  Classical post-net: {hqc_params['classical_post']:,}")
print(f"  Total:              {hqc_params['total']:,}")

print(f"\nClassical PINN:       {cpinn_params:,}")
print(f"\nReduction:            {1 - hqc_params['total']/cpinn_params:.1%}")

## 3. Data Preparation

In [ ]:
# Generate synthetic dataset (replace with real data for paper results)
dataset = FloodDataset.generate_synthetic(n_samples=1000, seed=42)

n_train = 700
n_val = 150
n_test = 150

train_data, val_data, test_data = random_split(
    dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)
test_loader = DataLoader(test_data, batch_size=32)

# Class distribution
counts = torch.bincount(dataset.targets, minlength=4)
print("=== Class Distribution ===")
for i, (name, count) in enumerate(zip(FLOOD_CLASSES.values(), counts)):
    print(f"  {name:20s}: {count:4d} ({count/len(dataset)*100:.1f}%)")

print(f"\nClass weights for focal loss: {dataset.class_weights().tolist()}")

## 4. Training with Physics-Informed Loss

The combined loss: L_HQC = L_focal + λ_SV · L_SV + λ_M · L_Manning

In [ ]:
# Setup training
loss_fn = PhysicsInformedLoss(
    lambda_sv=0.1,
    lambda_manning=0.05,
    focal_gamma=2.0,
    class_weights=dataset.class_weights(),
)

optimizer = torch.optim.Adam(hqc_model.parameters(), lr=1e-3, weight_decay=1e-5)

trainer = HQCPINNTrainer(
    model=hqc_model,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device='cpu',
    save_dir='../results/demo',
)

# Train (use more epochs for real experiments)
print("Training HQC-PINN (demo: 10 epochs)...")
history = trainer.train(
    train_loader, val_loader,
    n_epochs=10,
    patience=5,
)

print(f"\nTraining complete!")
print(f"Best validation loss: {history['best_val_loss']:.4f}")
print(f"Total epochs: {history['total_epochs']}")

## 5. Quantum Uncertainty Quantification

The inherent stochasticity of quantum measurement provides a natural UQ mechanism. Multiple measurement shots yield a distribution of predictions whose variance captures both aleatoric and epistemic uncertainty.

In [ ]:
# Get predictions with uncertainty
hqc_model.eval()
sample = torch.randn(5, TOTAL_FEATURES)  # 5 test samples

with torch.no_grad():
    results = hqc_model.predict_with_uncertainty(sample, n_shots=50)

print("=== Uncertainty Quantification Results ===")
for i in range(5):
    pred_class = results['probs_mean'][i].argmax().item()
    confidence = results['probs_mean'][i].max().item()
    entropy = results['entropy'][i].item()
    aleatoric = results['aleatoric_uncertainty'][i].item()
    
    print(f"  Sample {i}: Predicted={FLOOD_CLASSES[pred_class]}, "
          f"Confidence={confidence:.3f}, Entropy={entropy:.3f}, "
          f"Aleatoric={aleatoric:.4f}")

## 6. Results Summary

This demo uses synthetic data with limited epochs. See `run_experiment.py` for full paper results.

In [ ]:
# Evaluate on test set
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for batch in test_loader:
        logits = hqc_model(batch['features'])
        probs = torch.softmax(logits, dim=-1)
        all_preds.append(logits.argmax(dim=-1).numpy())
        all_targets.append(batch['targets'].numpy())
        all_probs.append(probs.numpy())

import numpy as np
y_pred = np.concatenate(all_preds)
y_true = np.concatenate(all_targets)
y_proba = np.concatenate(all_probs)

metrics = compute_classification_metrics(y_true, y_pred, y_proba)

print("=== Test Set Metrics ===")
print(f"  Accuracy:        {metrics['accuracy']:.4f}")
print(f"  F1 (macro):      {metrics['f1_macro']:.4f}")
print(f"  Precision:       {metrics['precision_macro']:.4f}")
print(f"  Recall:          {metrics['recall_macro']:.4f}")
if metrics.get('auc_roc'):
    print(f"  AUC-ROC:         {metrics['auc_roc']:.4f}")

print(f"\n{metrics['classification_report']}")